In [1]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
import sklearn

print("pandas version:", pd.__version__)
print("scikit-learn version:", sklearn.__version__)

# Step 2: Load the Telco Churn dataset directly from GitHub
DATA_URL = "https://raw.githubusercontent.com/carlosfab/dsnp2/master/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
df.head()

pandas version: 2.3.3
scikit-learn version: 1.7.2
Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Step 1: Check data types and missing values
df.info()

print("\n--- Missing values per column ---")
print(df.isnull().sum())

# Step 2: Check target variable distribution
print("\n--- Churn distribution ---")
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True) * 100)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
# Step 1: Convert TotalCharges to numeric (errors='coerce' turns bad values into NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Step 2: Check how many rows became NaN after conversion
print("Missing values in TotalCharges after conversion:", df['TotalCharges'].isnull().sum())

# Step 3: These are usually new customers with tenure = 0, fill with 0
df.loc[df['TotalCharges'].isnull(), 'tenure']  # check their tenure first
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Step 4: Drop customerID (not useful for prediction, it's just an identifier)
df = df.drop('customerID', axis=1)

# Step 5: Separate features (X) and target (y)
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})  # convert target to 0/1

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print(y.value_counts())

Missing values in TotalCharges after conversion: 11
Features shape: (7043, 19)
Target shape: (7043,)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

# Step 1: Split data BEFORE preprocessing (important! prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

# Step 2: Identify numeric and categorical columns automatically
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("\nNumeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

Training set shape: (5634, 19)
Test set shape: (1409, 19)

Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Step 1: Build preprocessing steps for each column type
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Step 2: Combine them into one ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Step 3: Build full pipeline = preprocessing + model
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Step 4: Train (fit) the pipeline
log_reg_pipeline.fit(X_train, y_train)

# Step 5: Predict and evaluate
y_pred = log_reg_pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

Accuracy: 0.8055358410220014

               precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [6]:
from sklearn.ensemble import RandomForestClassifier

# Step 1: Build Random Forest pipeline (same preprocessor, different model)
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Step 2: Train
rf_pipeline.fit(X_train, y_train)

# Step 3: Predict and evaluate
y_pred_rf = rf_pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\n", classification_report(y_test, y_pred_rf))

Accuracy: 0.7863733144073811

               precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.70      1409
weighted avg       0.77      0.79      0.78      1409



In [7]:
from sklearn.model_selection import GridSearchCV

# Step 1: Define hyperparameter grid for Logistic Regression
log_reg_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__class_weight': [None, 'balanced'],
    'classifier__solver': ['lbfgs', 'liblinear']
}

# Step 2: GridSearchCV - tries all combinations using cross-validation
log_reg_grid = GridSearchCV(
    log_reg_pipeline,
    param_grid=log_reg_param_grid,
    cv=5,
    scoring='f1',   # f1 score for churn class (better than accuracy for imbalanced data)
    n_jobs=-1
)

log_reg_grid.fit(X_train, y_train)

print("Best parameters:", log_reg_grid.best_params_)
print("Best CV F1 score:", log_reg_grid.best_score_)

# Step 3: Evaluate on test set
y_pred_best_lr = log_reg_grid.predict(X_test)
print("\nTest set performance:")
print(classification_report(y_test, y_pred_best_lr))

Best parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__solver': 'liblinear'}
Best CV F1 score: 0.6326714645434413

Test set performance:
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [8]:
# Step 1: Define hyperparameter grid for Random Forest
rf_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5],
    'classifier__class_weight': [None, 'balanced']
}

# Step 2: GridSearchCV
rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best CV F1 score:", rf_grid.best_score_)

# Step 3: Evaluate on test set
y_pred_best_rf = rf_grid.predict(X_test)
print("\nTest set performance:")
print(classification_report(y_test, y_pred_best_rf))

Best parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
Best CV F1 score: 0.6314195427892126

Test set performance:
              precision    recall  f1-score   support

           0       0.88      0.79      0.83      1035
           1       0.55      0.71      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.80      0.77      0.78      1409



In [9]:
import joblib
import os

# Step 1: Get the best Logistic Regression pipeline (already trained by GridSearchCV)
final_pipeline = log_reg_grid.best_estimator_

# Step 2: Save to a local non-synced path (avoid Anaconda Cloud sync issues)
output_dir = "C:/churn_pipeline_output"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "churn_prediction_pipeline.joblib")
joblib.dump(final_pipeline, output_path)

print("Pipeline saved to:", output_path)

# Step 3: Quick sanity check - load it back and test
loaded_pipeline = joblib.load(output_path)
sample_prediction = loaded_pipeline.predict(X_test.iloc[:5])
print("Sample predictions from loaded pipeline:", sample_prediction)
print("Actual values:", y_test.iloc[:5].values)

Pipeline saved to: C:/churn_pipeline_output\churn_prediction_pipeline.joblib
Sample predictions from loaded pipeline: [0 1 0 1 0]
Actual values: [0 0 0 0 0]


In [13]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# Load the trained pipeline
pipeline = joblib.load("churn_prediction_pipeline.joblib")

st.title("Customer Churn Prediction")
st.write("Enter customer details to predict if they are likely to churn.")

# Input fields matching our original features
col1, col2 = st.columns(2)

with col1:
    gender = st.selectbox("Gender", ["Male", "Female"])
    senior_citizen = st.selectbox("Senior Citizen", [0, 1])
    partner = st.selectbox("Partner", ["Yes", "No"])
    dependents = st.selectbox("Dependents", ["Yes", "No"])
    tenure = st.number_input("Tenure (months)", min_value=0, max_value=100, value=12)
    phone_service = st.selectbox("Phone Service", ["Yes", "No"])
    multiple_lines = st.selectbox("Multiple Lines", ["Yes", "No", "No phone service"])
    internet_service = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
    online_security = st.selectbox("Online Security", ["Yes", "No", "No internet service"])
    online_backup = st.selectbox("Online Backup", ["Yes", "No", "No internet service"])

with col2:
    device_protection = st.selectbox("Device Protection", ["Yes", "No", "No internet service"])
    tech_support = st.selectbox("Tech Support", ["Yes", "No", "No internet service"])
    streaming_tv = st.selectbox("Streaming TV", ["Yes", "No", "No internet service"])
    streaming_movies = st.selectbox("Streaming Movies", ["Yes", "No", "No internet service"])
    contract = st.selectbox("Contract", ["Month-to-month", "One year", "Two year"])
    paperless_billing = st.selectbox("Paperless Billing", ["Yes", "No"])
    payment_method = st.selectbox("Payment Method", ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"])
    monthly_charges = st.number_input("Monthly Charges", min_value=0.0, value=70.0)
    total_charges = st.number_input("Total Charges", min_value=0.0, value=840.0)

if st.button("Predict Churn"):
    # Build a single-row dataframe matching training data structure
    input_data = pd.DataFrame([{
        "gender": gender, "SeniorCitizen": senior_citizen, "Partner": partner,
        "Dependents": dependents, "tenure": tenure, "PhoneService": phone_service,
        "MultipleLines": multiple_lines, "InternetService": internet_service,
        "OnlineSecurity": online_security, "OnlineBackup": online_backup,
        "DeviceProtection": device_protection, "TechSupport": tech_support,
        "StreamingTV": streaming_tv, "StreamingMovies": streaming_movies,
        "Contract": contract, "PaperlessBilling": paperless_billing,
        "PaymentMethod": payment_method, "MonthlyCharges": monthly_charges,
        "TotalCharges": total_charges
    }])

    prediction = pipeline.predict(input_data)[0]
    probability = pipeline.predict_proba(input_data)[0][1]

    if prediction == 1:
        st.error(f"This customer is LIKELY TO CHURN (probability: {probability:.2%})")
    else:
        st.success(f"This customer is likely to STAY (probability of churn: {probability:.2%})")

Overwriting app.py


In [14]:
import shutil

# Step 1: Copy the joblib pipeline file into the current notebook's folder
# (app.py expects it in the same directory)
shutil.copy("C:/churn_pipeline_output/churn_prediction_pipeline.joblib", "churn_prediction_pipeline.joblib")

print("Pipeline copied to current folder.")

# Step 2: Create requirements.txt for GitHub / deployment
requirements = """streamlit
pandas
scikit-learn
joblib
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created.")

Pipeline copied to current folder.
requirements.txt created.


In [15]:
import os
print(os.getcwd())

C:\Users\zulno\anaconda_projects\418ebd50-80d4-4e70-bfbb-efad4386f1c8
